In [ ]:
!pip install -q open_clip_torch

In [ ]:
%cd /content

# 下载图片
!wget -O val2017.zip -q --show-progress http://images.cocodataset.org/zips/val2017.zip

# 下载标注
!wget -O annotations_trainval2017.zip -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# 解压
!unzip -o -q val2017.zip
!unzip -o -q annotations_trainval2017.zip

# 检查
print("图片数量:")
!find /content/val2017 -maxdepth 1 -name "*.jpg" | wc -l

print("标注文件:")
!ls -lh /content/annotations/captions_val2017.json

In [ ]:
import os
import json
import torch
import open_clip
from PIL import Image

# 选择设备
device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载模型和图片预处理
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32-quickgelu",
    pretrained="openai"
)

# 加载文本 tokenizer
tokenizer = open_clip.get_tokenizer("ViT-B-32-quickgelu")

model = model.to(device)
model.eval()

# 读取 COCO 标注
with open("/content/annotations/captions_val2017.json", "r", encoding="utf-8") as f:
    coco_data = json.load(f)

# 图片编号 -> 图片信息
images_by_id = {
    img["id"]: img
    for img in coco_data["images"]
}

# 图片编号 -> 描述列表
captions_by_image = {}
for ann in coco_data["annotations"]:
    captions_by_image.setdefault(
        ann["image_id"], []
    ).append(ann["caption"])

# 只选前 10 张图片
selected_image_ids = list(captions_by_image.keys())[:10]

images = []
texts = []

# 每张图片先只用第一条描述
for image_id in selected_image_ids:
    image_info = images_by_id[image_id]
    image_path = f"/content/val2017/{image_info['file_name']}"

    image = Image.open(image_path).convert("RGB")

    images.append(preprocess(image))
    texts.append(captions_by_image[image_id][0])

# 图片打包成张量
# 形状：[10, 3, 224, 224]
image_input = torch.stack(images).to(device)

# 文字转换成 token
# 形状：[10, token长度]
text_input = tokenizer(texts).to(device)

# 编码图片和文字
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_input)

    # 归一化
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # 相似度矩阵
    # 形状：[10, 10]
    similarity = image_features @ text_features.T

# 每张图片选出最匹配的文字
predicted_indices = similarity.argmax(dim=1)

# 正确答案就是图片自己的编号
correct_indices = torch.arange(10, device=device)

# 计算 Top-1 准确率
correct_count = (predicted_indices == correct_indices).sum().item()
accuracy = correct_count / 10

print("图片数量:", len(images))
print("文字数量:", len(texts))
print("相似度矩阵:", similarity.shape)
print("Top-1 准确率:", accuracy)
print("正确数量:", correct_count, "/ 10")

# 查看每张图片的预测
for i, text in enumerate(texts):
    predicted_text = texts[predicted_indices[i].item()]
    result = "正确" if i == predicted_indices[i].item() else "错误"

    print(f"\n图片 {i + 1}")
    print("正确描述:", text)
    print("预测描述:", predicted_text)
    print("结果:", result)

In [ ]:
print("image_input:", image_input.shape)
print("image_features:", image_features.shape)
print("text_features:", text_features.shape)
print("similarity:", similarity.shape)